TASK 1

#------------------------------------------------------------------------------------------------------------------------------------------#

Here I am importing needed modules and libraries.

In [3]:
from pathlib import Path

import pandas as pd
import numpy as np
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT, ISOLATION_LEVEL_SERIALIZABLE

Setting option for convenient work with pandas DataFrames.

In [4]:
pd.set_option('display.max_columns', None)

Writing down my password in the file not to make it easily available.

In [5]:
with open('pwd.txt', 'w') as file:
    file.write(input("Enter your password: "))

Enter your password:  Mz181007


Here I implemented a block for user's data

In [6]:
# Write here your data for Postgres:


with open('pwd.txt', 'r') as file:
    my_password = file.read().strip('\n')
my_host = 'localhost'
my_port = '5435'
my_database = 'app_db'
my_user_name = 'app'

path_for_images = '/Users/dariakozakevich/Desktop/ICEF/PYTHON/adv_pyt/archive-2/Housets_Image' #лапули тут пишем путь к картинкам из файла
path_for_database = '/Users/vsevolod/Downloads/HouseTS for Python project.csv' #path to the csv file

Now there are several functions to make it easier to work with SQL via Pandas and Python.

Function for connecting to a certain SQL database.

In [ ]:
def connection_to_database(pwd: str, host: str, user: str, db_name: str, port: str)-> psycopg2.extensions.connection | None:
    
    '''
    Connect to a db

    Inputs:
        usr_nm: str - username for the database
        host: str - host of the sql cluster/ DB
        port: str - target port
        db_nm: str - name of the database to connect to
        pwd: str - user password

    Outputs:
        psycopg2.extensions.connection - connection to the database
    '''

    conn = None

    try: 
        conn = psycopg2.connect(
            password = pwd,
            host = host,
            dbname = db_name,
            user = user,
            port = port
                               )
        
    except Exception as e:
        return f"There occurred the following error: {e}"
    
    else:
        print("Connection completed successfully!")

    return conn

Function for creating a new SQL database to work with the project.

In [ ]:
def creating_db_for_project(new_db_name: str,
              conn: psycopg2.extensions.connection | None = None,
              cur: psycopg2.extensions.cursor | None = None,
             ) -> None:
    
    '''
    Create a new db. For the argument AT LEAST ONE of the following variables should be specified: [conn, cur]. If only connection (maybe with cursor)
        is passed returns a tuple of the connection and cursor objects if only cursor is passed returns a cursor object

    Inputs:
        conn: psycopg2.extensions.connection - connection to a database
        cur: psycopg2.extensions.cursor - cursor in a database
        new_db_nm: str - the name of the database to be created

    Outputs:
        (psycopg2.extensions.connection, psycopg2.extensions.cursor) | psycopg2.extensions.cursor - used objects

    '''

    assert conn is not None or cur is not None, 'At least one of the variables should be passed!'
    
    conn_flag = True if conn is not None else False
    cur_flag = True if cur is not None else False


    if conn_flag == True and cur_flag == True:

        try:
            conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
            cur.execute(f"CREATE DATABASE {new_db_name}")
            conn.set_isolation_level(ISOLATION_LEVEL_SERIALIZABLE)

        except Exception as e:
            print(f"""
                    connection_passed: {conn_flag}
                    curcor_passed: {cur_flag}
                   """)
            return f"There occured the following error: {e}"
        
        else:
            print(f"Database {new_db_name} created successfully!")


    elif conn_flag == True and cur_flag == False:

        try:
            cur = conn.cursor()

            conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
            cur.execute(f"CREATE DATABASE {new_db_name}")
            conn.set_isolation_level(ISOLATION_LEVEL_SERIALIZABLE)

        except Exception as e:
            print(f"""
                    connection_passed: {conn_flag}
                    curcor_passed: {cur_flag}
                   """)
            return f"There occured the following error: {e}"
        
        else:
            print(f"Database {new_db_name} created successfully!")


    elif conn_flag == False and cur_flag == True:

        try:
            cur.execute(f"CREATE DATABASE {new_db_name}")

        except Exception as e:
            print(f"""
                    connection_passed: {conn_flag}
                    curcor_passed: {cur_flag}
                   """)
            return f"There occured the following error: {e}"
        
        else:
            print(f"Database {new_db_name} created successfully!")

    cur.close()
    conn.close()
            

Function to delete an SQL database if it is needed

In [ ]:
def delete_database(db_to_delete_name: str,
           conn: psycopg2.extensions.connection | None = None,
           cur: psycopg2.extensions.cursor | None = None,
          ) -> None | str:
    

    '''
    Delete a target database. For this function, AT LEAST ONE of the following variables should be specified: [conn, cur]. 
    If a connection is passed (optionally together with a cursor), the function uses it to execute the SQL command for deleting the database. 
    If only a cursor is passed, the function uses this cursor to execute the command.

    Inputs:
        db_to_delete_name: str - the name of the database to be deleted
        conn: psycopg2.extensions.connection - connection to a database
        cur: psycopg2.extensions.cursor - cursor in a database

    Outputs:
        None | str - prints a success message if the database is deleted successfully; otherwise returns an error message

    '''

    assert conn is not None or cur is not None, 'At least one of the variables should be passed!'
    
    conn_flag = True if conn is not None else False
    cur_flag = True if cur is not None else False

    if conn_flag == True and cur_flag == True:

        try:
            conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
            cur.execute(f"DROP DATABASE {db_to_delete_name}")
            conn.set_isolation_level(ISOLATION_LEVEL_SERIALIZABLE)

        except Exception as e:
            print(f"""
                    connection_passed: {conn_flag}
                    curcor_passed: {cur_flag}
                   """)
            return f"There occured the following error: {e}"
        
        else:
            print(f"Database {db_to_delete_name} deleted successfully!")

    elif conn_flag == True and cur_flag == False:

        try:
            cur = conn.cursor()

            conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
            cur.execute(f"DROP DATABASE {db_to_delete_name}")
            conn.set_isolation_level(ISOLATION_LEVEL_SERIALIZABLE)

        except Exception as e:
            print(f"""
                    connection_passed: {conn_flag}
                    curcor_passed: {cur_flag}
                   """)
            return f"There occured the following error: {e}"
        
        else:
            print(f"Database {db_to_delete_name} deleted successfully!")

    elif conn_flag == False and cur_flag == True:

        try:
            cur.execute(f"DROP DATABASE {db_to_delete_name}")
            conn.set_isolation_level(ISOLATION_LEVEL_SERIALIZABLE)
            
        except Exception as e:
            print(f"""
                    connection_passed: {conn_flag}
                    curcor_passed: {cur_flag}
                   """)
            return f"There occured the following error: {e}"
        else:
            print(f"Database {db_to_delete_name} deleted successfully!")

Function for creating a table in the SQL database.

In [ ]:
def create_table(table_name: str,
                 table_cols: list[str],
                 table_dtypes: list[str],
                 conn: psycopg2.extensions.connection | None = None,
                 cur: psycopg2.extensions.cursor | None = None,
                ) -> None | str :
    
    
    '''
    Create a new table in the target database. For this function, AT LEAST ONE of the following variables should be specified: [conn, cur]. 
    If a connection is passed (optionally together with a cursor), the function uses it to execute the SQL command for creating the table. 
    If only a cursor is passed, the function uses this cursor to execute the command.

    Inputs:
        table_name: str - the name of the table to be created
        table_cols: list[str] - the names of the table columns
        table_dtypes: list[str] - the data types of the table columns
        conn: psycopg2.extensions.connection - connection to a database
        cur: psycopg2.extensions.cursor - cursor in a database

    Outputs:
        None | str - prints a success message if the table is created successfully; otherwise returns an error message
    '''

    assert conn is not None or cur is not None, 'At least one of the variables should be passed!'

    conn_flag = True if conn is not None else False
    cur_flag = True if cur is not None else False
    
    # preparing the column definitions

    pairs = []
    for col, dtype in zip(table_cols, table_dtypes):
        pairs.append(' '.join([col, dtype]) + ',')
    pairs[-1] = pairs[-1].strip(',')
    columns = '\n'.join(pairs)


    
    if conn_flag == True and cur_flag == True:

        try:
            cur.execute(f"""CREATE TABLE IF NOT EXISTS {table_name} (
                    ID SERIAL,
                    {columns}
                    );
                        """)
            conn.commit()

        except Exception as e:
            print(f"""
                    connection_passed: {conn_flag}
                    curcor_passed: {cur_flag}
                   """)
            return f"There occurred the following error: {e}"
        
        else:
            print(f"Table {table_name} created successfully!")

    elif conn_flag == True and cur_flag == False:

        try:

            cur = conn.cursor()

            cur.execute(f"""CREATE TABLE IF NOT EXISTS {table_name} (
                    ID SERIAL,
                    {columns}
                    );
                        """)
            conn.commit()

        except Exception as e:
            print(f"""
                    connection_passed: {conn_flag}
                    curcor_passed: {cur_flag}
                   """)
            return f"There occurred the following error: {e}"
        
        else:
            print(f"Table {table_name} created successfully!")

    elif conn_flag == False and cur_flag == True:

        try:
            cur.execute(f"""CREATE TABLE IF NOT EXISTS {table_name} (
                    ID SERIAL,
                    {columns}
                    );
                        """)
            
            cur.connection.commit()

        except Exception as e:
            print(f"""
                    connection_passed: {conn_flag}
                    curcor_passed: {cur_flag}
                   """)
            return f"There occurred the following error: {e}"
        
        else:
            print(f"Table {table_name} created successfully!")

Function to delete a table in the SQL database.

In [ ]:
def delete_table(table_to_delete_name: str,
                 conn: psycopg2.extensions.connection | None = None,
                 cur: psycopg2.extensions.cursor | None = None,
                ) -> None | str:
    
    '''
    Delete a target table from the database. For this function, AT LEAST ONE of the following variables should be specified: [conn, cur]. 
    If a connection is passed (optionally together with a cursor), the function uses it to execute the SQL command for deleting the table. 
    If only a cursor is passed, the function uses this cursor to execute the command.

    Inputs:
        table_to_delete_name: str - the name of the table to be deleted
        conn: psycopg2.extensions.connection - connection to a database
        cur: psycopg2.extensions.cursor - cursor in a database

    Outputs:
        None | str - prints a success message if the table is deleted successfully; otherwise returns an error message
    '''

    assert conn is not None or cur is not None, 'At least one of the variables should be passed!'
    
    conn_flag = True if conn is not None else False
    cur_flag = True if cur is not None else False

    if conn_flag == True and cur_flag == True:

        try:
            cur.execute(f"DROP TABLE {table_to_delete_name}")
            conn.commit()

        except Exception as e:
            print(f"""
                    connection_passed: {conn_flag}
                    curcor_passed: {cur_flag}
                   """)
            return f"There occured the following error: {e}"
        
        else:
            print(f"Table {table_to_delete_name} deleted successfully!")

    elif conn_flag == True and cur_flag == False:

        try:
            cur = conn.cursor()
            cur.execute(f"DROP TABLE {table_to_delete_name}")
            conn.commit()

        except Exception as e:
            print(f"""
                    connection_passed: {conn_flag}
                    curcor_passed: {cur_flag}
                   """)
            return f"There occured the following error: {e}"
        
        else:
            print(f"Table {table_to_delete_name} deleted successfully!")

    elif conn_flag == False and cur_flag == True:

        try:
            cur.execute(f"DROP TABLE {table_to_delete_name}")
            cur.connection.commit()
            
        except Exception as e:
            print(f"""
                    connection_passed: {conn_flag}
                    curcor_passed: {cur_flag}
                   """)
            return f"There occured the following error: {e}"
        else:
            print(f"Table {table_to_delete_name} deleted successfully!")


One of the main sections: via this function I uppload data from .csv file after reading it in Pandas and pictures from path.

In [ ]:
def upload_data_to_table(table_name: str,
                         df: pd.DataFrame,
                         zip_codes: pd.Series | None = None,
                         conn: psycopg2.extensions.connection | None = None,
                         cur: psycopg2.extensions.cursor | None = None
                        ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor]| (psycopg2.extensions.cursor):

    assert conn is not None or cur is not None, 'At least one of the variables should be passed!'

    conn_flag = True if conn is not None else False
    cur_flag = True if cur is not None else False


    '''
    Upload data from a pandas DataFrame to a target table in the database. 
    For the argument AT LEAST ONE of the following variables should be specified: [conn, cur]. 
    If only connection (maybe with cursor) is passed returns a tuple of the connection and cursor objects; if only cursor is passed returns a cursor object.

    Inputs:
        table_name: str - the name of the table to which the data will be uploaded
        df: pd.DataFrame - the DataFrame containing the data to be inserted into the table
        zip_codes: pd.Series | None - optional Series with zip codes
        conn: psycopg2.extensions.connection - connection to a database
        cur: psycopg2.extensions.cursor - cursor in a database

    Outputs:
        tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | psycopg2.extensions.cursor - used objects
    '''

    
    if conn_flag == True and cur_flag == True:

        try:
            conn.autocommit = True

            for idx, data in df.iterrows():
                cur.execute(f"INSERT INTO {table_name} ({', '.join([d.strip(' ').replace(" ", '_') for d in df.columns])}) VALUES ({', '.join(['%s'] * df.shape[1])});", tuple(data))

        except Exception as e:
            raise e

        
            
    if conn_flag == True and cur_flag == False:

        try:
            conn.autocommit = True
            cur = conn.cursor()

            for idx, data in df.iterrows():
                cur.execute(f"INSERT INTO {table_name} ({', '.join([d.strip(' ').replace(" ", '_') for d in df.columns])}) VALUES ({', '.join(['%s'] * df.shape[1])});", tuple(data))
        
        except Exception as e:
            raise e
            
    if conn_flag == False and cur_flag == True:

        conn.autocommit = True
        try:
            for idx, data in df.iterrows():
                cur.execute(f"INSERT INTO {table_name} ({', '.join([d.strip(' ').replace(" ", '_') for d in df.columns])}) VALUES ({', '.join(['%s'] * df.shape[1])});", tuple(data))
        
        except Exception as e:
            raise e
    
    if conn_flag == True:
        return (conn, cur)
    else:
        return cur            

Function for "giving life" for table with information from .csv file and for table with photos.

In [ ]:
def pognaly_delat_gryaz(pwd: str,
                        host: str,
                        user: str,
                        db_name: str,
                        port: str
                       )-> (psycopg2.extensions.connection, psycopg2.extensions.cursor):


        
    conn = connection_to_database(pwd = pwd,              # Connecting to project database
                                  host = host,
                                  user = user,
                                  db_name = db_name,
                                  port = port
                                 )
    cur = conn.cursor()
    

    ''' Prepare the schema for the SQL data table '''

    # only first row - just to inspect the column structure and infer the data types
    df_temporarily = pd.read_csv(path_for_database, nrows = 1) 

    cols_for_sql_table = [stroka.strip(' ').replace(" ", '_') for stroka in df_temporarily.columns]    # list of name of columns for SQL table
    dtypes = [str(dtype) for dtype in df_temporarily.dtypes]    # list of pandas types (in str)



    mapping = {"object": 'TEXT',    # match to SQL dtypes
               "float64": 'REAL',
               "int64": 'INTEGER',
               "str": 'TEXT'
              }
    dtypes_in_sql_dtypes = [mapping[dtype] for dtype in dtypes]

    # create table "housets"
    table_name = 'housets'
    create_table(table_name = table_name, table_cols = cols_for_sql_table, table_dtypes = dtypes_in_sql_dtypes, conn = conn)



    ''' And now we are starting to filling the SQL table with rows from the given csv file: '''


    print('Starting to inserting data: \n\n')


    for idx, chunk_of_df in enumerate(pd.read_csv(path_for_database, chunksize = 10_000)):     # insert data to the table by chunks since it is a big dataframe

        upload_data_to_table(conn = conn, 
                             table_name = table_name,
                             df = chunk_of_df
                            )
        print(f'Chunk {idx + 1} is inserted.\n')

    print(f'All data is inserted into the table: {table_name}. Total numbers of chunks used = {idx + 1}.\n\n')




    ''' Preparing for the table with images'''

    my_path = path_for_images
    downloads = Path(my_path)


    housets_final = downloads / "Housets_Final"    # check if Housets_Final is there
    images_with_paths = [photo for photo in housets_final.rglob("*") if photo.suffix.lower() == ".png"]     # look inside all of our documents and seek only for pngs


    # make a list with names and pathes to photos:

    images_just_names = [photo.stem for photo in images_with_paths] 
    images_with_paths = [str(photo) for photo in images_with_paths]

    data_for_df_with_photos = []
    for image_name, image_path in zip(images_just_names, images_with_paths):
        data_for_df_with_photos.append((image_name, image_path))

    # make a pdDaraFrame from list
    df_with_photos = pd.DataFrame(data = data_for_df_with_photos, columns = ['names_of_files', 'paths'])
    print(df_with_photos)
    
    ''' Prepare the schema for the SQL images table '''

    cols_for_sql_table = [stroka.strip(' ').replace(" ", '_') for stroka in df_with_photos.columns]
    dtypes = [str(dtype) for dtype in df_with_photos.dtypes]   # We may not convert it to list
    dtypes_in_sql_dtypes = [mapping[dtype] for dtype in dtypes]

    table_name_for_photos = 'photos_for_housets'
    create_table(table_name = table_name_for_photos, table_cols = cols_for_sql_table, table_dtypes = dtypes_in_sql_dtypes, conn = conn)


    ''' And now we are starting to filling the SQL table with rows from the given csv file '''


    print('Starting to inserting data: \n\n')
    
    upload_data_to_table(conn = conn,
                         table_name = table_name_for_photos,
                         df = df_with_photos
                        )
    
    print(f'All data is inserted into the table: {table_name_for_photos}')
  
    return (conn, cur)

```TASK 1```

Import the HouseTS tabular panel with 6000 ZIP codes, 39 columns, and monthly data from 2012-2023. Integrate with provided aerial imagery metadata and socioeconomic indicators in the way you see fit (for example use path to the imagery). Create consistent ZIP-code-year-month keys.

Here I start all functions in the right order to 1) Delete database with tables if the program was runned before,
                                                 2) Create the database for project,
                                                 3) Create the needed tables and fill them with appropriate data.

Connecting to the initial database ```postgres```


In [ ]:
conn = connection_to_database(pwd = my_password, host = my_host, user = my_user_name, db_name = my_database, port = my_port)

Deleting the database```python_project_real_estate``` for project to be sure that it doesn't exist

In [ ]:
delete_database(db_to_delete_name = 'python_project_real_estate', conn = conn)

Creating a database for project and closing the conndection and cursor for the initial database 

In [ ]:
creating_db_for_project(new_db_name = 'python_project_real_estate', conn = conn)

Connecting to a database ```python_project_real_estate```, creating a cursor, creating appropriate tables and filling them with data

In [ ]:
conn, cur = pognaly_delat_gryaz(my_password, my_host, my_user_name, 'python_project_real_estate', my_port) 

This code first changes the table `housets` by adding new columnы called `zipcode_year_month` (as it was asked in the task) and `zipcode_year` (to be able to merge photos) if they does not already exist. Then it fills these columns for all rows by combining the `zipcode` value with the first 4 characters of the `date` column (year) and substring of the `date` (month) and `zipcode` value with the first 4 characters of the `date` column (year) for `zipcode_year`.


In [ ]:
cur.execute("""ALTER TABLE housets
               ADD COLUMN IF NOT EXISTS
               zipcode_year_month VARCHAR(50)""")

cur.execute("""ALTER TABLE housets
               ADD COLUMN IF NOT EXISTS
               zipcode_year VARCHAR(50)""")



cur.execute("""UPDATE housets
               SET zipcode_year_month = zipcode::VARCHAR(50) || '_' ||
               LEFT("date"::VARCHAR(50), 4) || '_' ||
               SUBSTRING("date"::VARCHAR(50) FROM 6 FOR 2)""")

cur.execute("""UPDATE housets
               SET zipcode_year = zipcode::VARCHAR(50) || '_' || LEFT("date"::VARCHAR(50), 4);""")



conn.commit()

In [ ]:
import os
os.getcwd()

Some work with Pandas DataFrame just to see the .csv file readed with a new column "zipcode_year", our SQL table should look the same

In [ ]:
list_of_pieces_of_df = []

for idx, df in enumerate(pd.read_csv('HouseTS for Python project.csv', chunksize = 10_000)):
    list_of_pieces_of_df.append(df)

df_housets = pd.concat(list_of_pieces_of_df, ignore_index = True)


df_housets['zipcode_year'] = df_housets['zipcode'].astype('str') + '_' + pd.to_datetime(df_housets['date']).dt.strftime('%Y').astype('str')
df_housets['zipcode_year_month'] = df_housets['zipcode'].astype('str') + '_' + pd.to_datetime(df_housets['date']).dt.strftime('%Y-%m').astype('str')

df_housets

Showing columns and the first 10 rows of the 2 filled tables, converting them from SQL tables into Pandas DataFrames

In [ ]:
cur.execute("SELECT * FROM housets LIMIT 10;")

# Fetch all rows as a list of tuples
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
print(columns, "\n")
print(pd.DataFrame(data = data, columns = columns), "\n\n\n")


cur.execute("SELECT * FROM photos_for_housets LIMIT 10;")
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
print(columns, "\n")
print(pd.DataFrame(data = data, columns = columns))

Now let's join the two tables via SQL query

In [ ]:


cur.execute('''
CREATE TABLE IF NOT EXISTS combined_housets AS
SELECT 
housets.*,           
photos_for_housets.names_of_files,
photos_for_housets.paths
FROM housets
LEFT JOIN photos_for_housets
ON housets.zipcode_year = photos_for_housets.names_of_files;
''')

conn.commit()

Showing the new combined SQL table via convering it into Pandas DataFrame

In [ ]:
cur.execute("""
SELECT * FROM combined_housets
""")

data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
print(columns)
print(pd.DataFrame(data = data, columns = columns))

Selecting only such rows from the combined table ```combined_houdets```, where columns ```paths``` and ```names_of_files``` are filled, than taking them as data for Pandas DataFrame to display it

In [ ]:
cur.execute("""SELECT * FROM combined_housets
               WHERE paths IS NOT NULL
               AND names_of_files IS NOT NULL""")

data = cur.fetchall()
columns = [desc[0] for desc in cur.description]

print(pd.DataFrame(data = data, columns = columns))

Counting the number of rows with appropriate photos, displaying it via Pandas DataFrame and closing the cursor and connection

In [ ]:
cur.execute("""SELECT COUNT(*) FROM combined_housets
                WHERE paths IS NOT NULL
                AND names_of_files IS NOT NULL""")

data = cur.fetchall()
print(f"Number of rows where column 'path' is not None: {data[0][0]}")

#------------------------------------------------------------------------------------------------------------------------------------------#

```TASK 2```

ZHVI (Zillow Home Value Index) is the primary target. Create derived price metrics: price per square foot (using property size data), median listing price, and sale-to-list ratio

In [ ]:
path_for_cpi = "/Users/vsevolod/Downloads/CPIAUCSL.csv"

Here we construct a base for our future CPI_lookup_table using the data from https://fred.stlouisfed.org/series/CPIAUCSL, making the March of 2012 the base month for our index (take its value as 100). For convenience we also construct two new columns "year" and "month".

In [ ]:
df_cpi = pd.read_csv(path_for_cpi)
df_cpi

In [ ]:
df_cpi = pd.read_csv(path_for_cpi).rename(columns = {"observation_date": "date"}).rename(columns = {"CPIAUCSL": "cpi"})

df_cpi["year"] = pd.to_datetime(df_cpi["date"]).dt.strftime('%Y')
df_cpi["month"] = pd.to_datetime(df_cpi["date"]).dt.strftime('%m')
df_cpi = df_cpi.drop(columns = ["date"])

# take base year 2012
base_2012_03 = df_cpi.loc[
    (df_cpi["year"] == "2012") & (df_cpi["month"] == "03"),
    "cpi"
].iloc[0]

for i in range(len(df_cpi["cpi"])):
    df_cpi.loc[i, "cpi"] = round((df_cpi.loc[i, "cpi"]/base_2012_03) * 100, 3)

df_cpi

Here we create the table CPI_lookup using SQL and fill it with the values from the df_cpi created earlier

In [ ]:
table_columns = df_cpi.columns # column names


# convert types for SQL
dtypes = [str(dtype) for dtype in df_cpi.dtypes]
mapping = { "object": 'TEXT',
            "float64": 'REAL',
            "str" : "TEXT" 
            }
table_dtypes = [mapping[dtype] for dtype in dtypes]

create_table(table_name = 'cpi_lookup', table_cols = table_columns, table_dtypes = table_dtypes, conn = conn)

print('Starting to inserting data: \n\n')
upload_data_to_table(conn = conn,
                         table_name = 'cpi_lookup',
                         df = df_cpi
                        )
    
print(f'All data is inserted into the table: {'cpi_lookup'}')

# show the table

cur.execute('''SELECT cpi, year, month FROM cpi_lookup''')
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_show = pd.DataFrame(data = data, columns = columns)
df_show

In [ ]:
cur.execute("""UPDATE cpi_lookup
SET month = month::INTEGER::TEXT;""") 
# to remove leading zeros


cur.execute("""SELECT * FROM cpi_lookup""")
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_show = pd.DataFrame(data = data, columns = columns)
df_show

Here we prepare our cpi_lookup table to join it with the housets table. (Why do we use this method is explained in the cell below)

In [ ]:
# add column cpi
cur.execute("""ALTER TABLE combined_housets ADD COLUMN IF NOT EXISTS cpi FLOAT;""")

# join: take each row in the housets table, find the CPI that matches the year and month from that row’s date and write that CPI into the cpi column
cur.execute("""UPDATE combined_housets h
        SET cpi = c.cpi 
        FROM (SELECT DISTINCT ON (year, month) year, month, cpi FROM cpi_lookup) c 
        WHERE EXTRACT(year FROM date::DATE)::TEXT = c.year AND EXTRACT(month FROM date::DATE)::TEXT = c.month;""")
conn.commit()

# show the data 

cur.execute("""SELECT * FROM combined_housets LIMIT 5;""")
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
print("combined_housets dates:")
df_show = pd.DataFrame(data=data, columns=columns)
df_show


The problem was: when using EXTRACT for 'year' and 'month' to compare it with 'year' and 'month' in the cpi_lookup table, month values become '3' instead of '03', that is why it would be easier to just change the data display in the cpi_lookup.month to insert a column to housets

Here we create a median_list_price adjusted to level out the inflation rate. Basically, a real median listing price for each property

In [ ]:
# add column 'median_list_price_inf' to the housets table
cur.execute("""ALTER TABLE combined_housets
ADD COLUMN IF NOT EXISTS median_list_price_inf FLOAT;""")

# inflation adjustment
cur.execute("""UPDATE combined_housets h 
SET median_list_price_inf = h.median_list_price * 100/NULLIF(h.cpi, 0);""")

conn.commit()

cur.execute("""SELECT * FROM combined_housets""")
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_show = pd.DataFrame(data = data, columns = columns)
df_show

Some graphs just for fun:

In [ ]:
import matplotlib.pyplot as plt

# make sure date is datetime
df_show['date'] = pd.to_datetime(df_show['date'])

# extract year
df_show['year'] = df_show['date'].dt.year

# aggregate by year
df_yearly = (
    df_show
    .groupby('year', as_index=False)[['median_list_price', 'median_list_price_inf']]
    .mean()
)

# plot
plt.figure(figsize=(12, 6))
plt.plot(df_yearly['year'], df_yearly['median_list_price'], label='median_list_price')
plt.plot(df_yearly['year'], df_yearly['median_list_price_inf'], label='median_list_price_inf')

plt.title('Median List Price vs Inflation-Adjusted Median List Price by Year')
plt.xlabel('Year')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.show()

Here we create a janky analogue of ZHVI using median_home_value: "2012-03-31" is the date of the base value for each zipcode
Why don't we use a unified base period? -> we need to be able to compare the growth rate of the home values between zipcodes, therefore, it is more convenient to construct separate "system" of indices for each zipcode

In [ ]:
cur.execute("""ALTER TABLE combined_housets
ADD COLUMN IF NOT EXISTS zhvi_analogue_mhv FLOAT;""")

cur.execute("""UPDATE combined_housets h
SET zhvi_analogue_mhv = h.median_home_value * 100/NULLIF(base.median_home_value,0)
FROM combined_housets base
WHERE h.zipcode = base.zipcode 
AND base.date::DATE = '2012-03-31';""")

conn.commit()

Here we construct a new sale_to_list ratio (we already have avg_sale_to_list) out of the given median values.

In [ ]:
cur.execute("""ALTER TABLE combined_housets
ADD COLUMN IF NOT EXISTS derived__median_sale_to_list FLOAT;""")

cur.execute("""UPDATE combined_housets h 
SET derived__median_sale_to_list = h.median_sale_price/NULLIF(h.median_list_price, 0);""")
conn.commit()

cur.execute("""SELECT * FROM combined_housets ORDER BY zipcode ASC, date ASC""")
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_show = pd.DataFrame(data = data, columns = columns)
df_show

In this cell we construct a price per square foot metrics using the provided median_ppsf metrics. We adjust it to level out the inflation using cpi.

In [ ]:
cur.execute("""ALTER TABLE combined_housets
ADD COLUMN IF NOT EXISTS median_ppsf_inf FLOAT;""")

cur.execute("""UPDATE combined_housets h
SET median_ppsf_inf = h.median_ppsf * 100/NULLIF(h.cpi, 0);""")
conn.commit()

cur.execute("""SELECT * FROM combined_housets;""")
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_show = pd.DataFrame(data = data, columns = columns)
df_show

```TASK 3```

Calculate months of supply (inventory / monthly sales pace), new listings penetration (new listings / total inventory), and absorption rate (sales / inventory)

In [ ]:
cur.execute("""
    ALTER TABLE combined_housets
    ADD COLUMN IF NOT EXISTS months_of_supply REAL
""")

cur.execute("""UPDATE combined_housets h
SET months_of_supply = inventory::REAL / NULLIF(homes_sold, 0);""")

conn.commit()

cur.execute("""SELECT * FROM combined_housets LIMIT 10;""")
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_show = pd.DataFrame(data = data, columns = columns)
df_show

In [ ]:
cur.execute("""
    ALTER TABLE combined_housets
    ADD COLUMN IF NOT EXISTS new_listings_penetration REAL
""")

cur.execute("""UPDATE combined_housets h
SET new_listings_penetration = new_listings::REAL / NULLIF(inventory, 0);""")

conn.commit()

cur.execute("""SELECT * FROM combined_housets LIMIT 10;""")
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_show = pd.DataFrame(data = data, columns = columns)
df_show

In [ ]:
cur.execute("""
    ALTER TABLE combined_housets
    ADD COLUMN IF NOT EXISTS absorption_rate REAL
""")

cur.execute("""UPDATE combined_housets h
SET absorption_rate = homes_sold::REAL / NULLIF(inventory, 0);""")

conn.commit()

cur.execute("""SELECT * FROM combined_housets LIMIT 10;""")
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_show = pd.DataFrame(data = data, columns = columns)
df_show

In [ ]:
cur.execute("""
    ALTER TABLE combined_housets
    ADD COLUMN IF NOT EXISTS market_type TEXT
""")

cur.execute("""
    UPDATE combined_housets
    SET market_type = CASE
            WHEN absorption_rate >= 0.20 THEN 'seller_market'
            WHEN absorption_rate >= 0.10 THEN 'balanced_market'
            WHEN absorption_rate < 0.10 THEN 'buyer_market'
        ELSE NULL
    END
""")

conn.commit()


cur.execute("""SELECT * FROM combined_housets""")
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_show = pd.DataFrame(data = data, columns = columns)
df_show

```TASK 4 ```

Calculate price changes over multiple horizons: 1-month, 3-month, 6-month, 12-month, and 3-year. Annualize each return for comparison. Calculate price acceleration (change in momentum) as 3-month change minus previous 3-month change.

We simply create all the needed columns to fill them with values later

In [ ]:
cur.execute("""ALTER TABLE combined_housets ADD COLUMN IF NOT EXISTS return_1m FLOAT;""")
cur.execute("""ALTER TABLE combined_housets ADD COLUMN IF NOT EXISTS return_3m FLOAT;""")
cur.execute("""ALTER TABLE combined_housets ADD COLUMN IF NOT EXISTS return_6m FLOAT;""")
cur.execute("""ALTER TABLE combined_housets ADD COLUMN IF NOT EXISTS return_12m FLOAT;""")
cur.execute("""ALTER TABLE combined_housets ADD COLUMN IF NOT EXISTS return_36m FLOAT;""")
cur.execute("""ALTER TABLE combined_housets ADD COLUMN IF NOT EXISTS return_1m_ann FLOAT;""")
cur.execute("""ALTER TABLE combined_housets ADD COLUMN IF NOT EXISTS return_3m_ann FLOAT;""")
cur.execute("""ALTER TABLE combined_housets ADD COLUMN IF NOT EXISTS return_6m_ann FLOAT;""")
cur.execute("""ALTER TABLE combined_housets ADD COLUMN IF NOT EXISTS return_12m_ann FLOAT;""")
cur.execute("""ALTER TABLE combined_housets ADD COLUMN IF NOT EXISTS return_36m_ann FLOAT;""")
cur.execute("""ALTER TABLE combined_housets ADD COLUMN IF NOT EXISTS price_acceleration FLOAT;""")

Here we use the thing that goes beyond our course program - Common Table Expressions for the simplicity of reading the code. To construct returns we need to pass the parameter PARTITION BY to correctly find previous prices for each separate zipcode.

For convenient display we need to sort the table rows first by zipcode, then by date. NaN values are perfectly normal for our result table, as we don't have the values for the first n-elements in each zipcode (depending on the number of months in a period)

*All returns computed in percentage values

*Of course we could have used the subqueries, but they are far less readable than CTE method

In [ ]:
cur.execute("""WITH lags AS (SELECT id, price,
              LAG(price, 1) OVER (PARTITION BY zipcode ORDER BY date::DATE) AS lag_1m,
              LAG(price, 3) OVER (PARTITION BY zipcode ORDER BY date::DATE) AS lag_3m,
              LAG(price, 6) OVER (PARTITION BY zipcode ORDER BY date::DATE) AS lag_6m,
              LAG(price, 12) OVER (PARTITION BY zipcode ORDER BY date::DATE) AS lag_12m,
              LAG(price, 36) OVER (PARTITION BY zipcode ORDER BY date::DATE) AS lag_36m FROM housets),
returns AS (SELECT id,
            (price - lag_1m)/NULLIF(lag_1m, 0) AS ret_1m,
            (price - lag_3m)/NULLIF(lag_3m, 0) AS ret_3m,
            (price - lag_6m)/NULLIF(lag_6m, 0) AS ret_6m,
            (price - lag_12m)/NULLIF(lag_12m, 0) AS ret_12m,
            (price - lag_36m)/NULLIF(lag_36m, 0) AS ret_36m
            FROM lags)
                 
UPDATE combined_housets h
SET return_1m = re.ret_1m*100,
    return_3m = re.ret_3m*100,
    return_6m = re.ret_6m*100,
    return_12m = re.ret_12m*100,
    return_36m = re.ret_36m*100,
    return_1m_ann = (POWER(1 + re.ret_1m, 12/1) - 1)*100,
    return_3m_ann = (POWER(1 + re.ret_3m, 12/3) - 1)*100,
    return_6m_ann = (POWER(1 + re.ret_6m, 12/6) - 1)*100,
    return_12m_ann = (POWER(1 + re.ret_12m, 12/12) - 1)*100,
    return_36m_ann = (POWER(1 + re.ret_36m, 12.0/36) - 1)*100
    FROM returns re
WHERE h.id = re.id;""")

conn.commit()

cur.execute("""SELECT * FROM combined_housets ORDER BY zipcode ASC, date ASC LIMIT 38""")
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
print(pd.DataFrame(data = data, columns = columns))

Here we compute price change acceleration (percentage) also using CTEs
We used LAG(price, 6) as it basically means LAG(LAG(price, 3), 3), therefore, we compute the change in previous momentum


In [ ]:
cur.execute("""WITH acceleration AS (SELECT id,
                    (price - LAG(price, 3) OVER (PARTITION BY zipcode ORDER BY date::DATE))*100/NULLIF(LAG(price, 3) OVER (PARTITION BY zipcode ORDER BY date::DATE), 0)
                      AS current_momentum,
                    (LAG(price, 3) OVER (PARTITION BY zipcode ORDER BY date::DATE) - LAG(price, 6) OVER (PARTITION BY zipcode ORDER BY date::DATE))*100/NULLIF(LAG(price, 6) OVER (PARTITION BY zipcode ORDER BY date::DATE), 0) 
                      AS previous_momentum FROM combined_housets)
UPDATE combined_housets h
SET price_acceleration = current_momentum - previous_momentum
FROM acceleration a
WHERE h.id = a.id;""")

conn.commit()

cur.execute("""SELECT * FROM combined_housets ORDER BY zipcode ASC, date ASC LIMIT 10""")
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
print(pd.DataFrame(data = data, columns = columns))

In [ ]:
#for those who will come after me
cur.execute("""SELECT * FROM combined_housets ORDER BY zipcode ASC, date ASC""")
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df = pd.DataFrame(data = data, columns = columns)
df

TASK 5


In [ ]:
cur.execute("""SELECT * FROM combined_housets""")
columns = [desc[0] for desc in cur.description]
columns

Let's add the necessary columns to the general table: price_vol_12m, downside_risk_12m, risk_adjusted_return_12m

In [ ]:
cur.execute(""" ALTER TABLE combined_housets 
ADD COLUMN IF NOT EXISTS price_vol_12m FLOAT,
ADD COLUMN IF NOT EXISTS downside_risk_12m FLOAT,
ADD COLUMN IF NOT EXISTS risk_adjusted_return_12m FLOAT """) 

conn.commit()
# ALTER TABLE - change a table without deleting and recreating it

You need to calculate the standard deviation for the previous 12 months using the return_1m column (separately for each zip code)

In [ ]:
cur.execute(""" UPDATE combined_housets
                SET price_vol_12m = temp_table.price_vol_12m
                FROM  (
                    SELECT
                          id, 
                          CASE
                                WHEN COUNT(return_1m) OVER (
                                    PARTITION BY zipcode
                                    ORDER BY date
                                    ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
                                ) = 12
                                THEN STDDEV(return_1m) OVER (
                                    PARTITION BY zipcode
                                    ORDER BY date
                                    ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
                                )
                                ELSE NULL
                            END AS price_vol_12m
                        FROM combined_housets
                ) temp_table
                WHERE combined_housets.id = temp_table.id
                """)

conn.commit()


# UPDATE housets --- update our table
# SET --- write to the required column of our table the calculated column from the temporary (auxiliary) table

# [SET] FROM (SELECT ... FROM housets -- from each row of our table take id and calculate something using CASE) temp_table --- creates a temporary table temp_table inside the query (we give it an alias to reference it later),
# in which the calculations will be performed

# SELECT id - so that later we can match each calculated row with the value in our table
# that is, for now the temp_table contains just id

# CASE --- END | start -(condition)- end 
# WHEN --- if COUNT in the selected "window" () equals 12, meaning there are 12 return_1m values in the selected 12-period window
# THEN --- then calculate sigma in the selected "window"
# ELSE NULL --- since we are looking at a full year

# OVER --- we specify relative to which rows COUNT will operate
    # PARTITION BY zipcode --- we need to calculate for each house separately, meaning the window takes all rows for each zipcode
    # ORDER BY date --- sort by date just in case
    # ROWS ... --- take 11 previous rows and the current one

# now, when we have 12 records, we apply stddev to the same window, and if there are not 12, we write NULL

# as a result, we calculated price_vol_12m for each id
# END AS --- assign a name to the calculated column

# SET ... FROM ... WHERE housets.id = temp_table.id -- set values in the required column, taking them from the temporary table where ids match (i.e., to align rows)


In [ ]:
cur.execute("""SELECT return_1m 
FROM combined_housets
WHERE price_vol_12m IS NOT NULL""")
len(cur.fetchall())

Downside risk - the same, but only for negative return_1m


In [ ]:
cur.execute(""" UPDATE combined_housets
                SET downside_risk_12m = temp_table.downside_risk_12m
                FROM  (
                    SELECT
                          id, 
                          CASE
                                WHEN COUNT(return_1m) OVER (
                                    PARTITION BY zipcode
                                    ORDER BY date
                                    ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
                                ) = 12
                                THEN STDDEV(
                                    CASE
                                        WHEN return_1m < 0 THEN return_1m
                                        ELSE NULL
                                    END
                                ) OVER (
                                    PARTITION BY zipcode
                                    ORDER BY date
                                    ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
                                )
                                ELSE NULL
                            END AS downside_risk_12m
                        FROM combined_housets
                ) temp_table
                WHERE combined_housets.id = temp_table.id
                """)

conn.commit()

# simply add another CASE to check return_1m < 0
# that is, we take the same full window and within it calculate only for negative returns
# once again: first we take the window and ensure it contains 12 values, and then apply stddev to that same window, but only to returns < 0

In [ ]:
cur.execute("""SELECT return_1m 
FROM combined_housets
WHERE downside_risk_12m IS NOT NULL""")
len(cur.fetchall())

We need to calculate risk_adjusted_return_12m, which is (quote from gpt): how much return we get per unit of risk. RAR = AVG_return_1m / sigma(return_1m)

In [ ]:
cur.execute(""" UPDATE combined_housets
                SET risk_adjusted_return_12m = temp_table.risk_adjusted_return_12m
                FROM  (
                    SELECT
                          id, 
                          CASE
                                WHEN COUNT(return_1m) OVER (
                                    PARTITION BY zipcode
                                    ORDER BY date
                                    ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
                                ) = 12
                                THEN CASE
                                        WHEN STDDEV(return_1m) OVER (
                                            PARTITION BY zipcode
                                            ORDER BY date
                                            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
                                        ) = 0
                                        THEN NULL
                                        ELSE AVG(return_1m) OVER (
                                            PARTITION BY zipcode
                                            ORDER BY date
                                            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
                                        ) 
                                        
                                        / 

                                        STDDEV(return_1m) OVER (
                                            PARTITION BY zipcode
                                            ORDER BY date
                                            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
                                        )
                                     END
                                ELSE NULL
                            END AS risk_adjusted_return_12m
                        FROM combined_housets
                ) temp_table
                WHERE combined_housets.id = temp_table.id
                """)

conn.commit()


# again, nothing has changed, we just now check that sigma in the window is not zero, i.e., to avoid division by zero
# if sigma = 0, then we write NULL

In [ ]:
cur.execute("""SELECT return_1m 
FROM combined_housets
WHERE risk_adjusted_return_12m IS NOT NULL""")
len(cur.fetchall())

In [ ]:
cur.execute("""SELECT * FROM combined_housets ORDER BY zipcode ASC, date ASC""")
data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df = pd.DataFrame(data = data, columns = columns)
df

```TASK 6```


For each ZIP code, calculate metrics relative to its metropolitan area: price premium (ZIP price / metro median), inventory share, and price momentum rank within metro. Identify ZIP codes leading or lagging the metro trend.

just checking the data if everything alright (+ to show with what data i am going to work)

In [ ]:
cur.execute("""
    SELECT
        date,
        zipcode,
        city_full,
        price,
        inventory
    FROM combined_housets
    WHERE city_full IS NOT NULL
      AND zipcode IS NOT NULL
      AND price IS NOT NULL
      AND inventory IS NOT NULL
    LIMIT 10;
""")

data = cur.fetchall()
columns = [desc[0] for desc in cur.description]

df_show = pd.DataFrame(data=data, columns=columns)
df_show

Here we make 2 new variables with data: ```metro_median_price``` and ```metro_total_inventory``` that i will later use for solutions

For ```metro_median_price``` I use ```PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY price)``` to find median price and ```(PARTITION BY city_full, date)``` to find this price only among unique metro-regions and data

For ```metro_total_inventory``` I use ```SUM(inventory)``` and ```(PARTITION BY city_full, date)``` to find total inventory among the same parameters as above

I coulnd't use here PARTITION BY since function PERCENTILE_CONT needs the use of GROUP BY and not window syntax

In [ ]:
# i drop metro_stats_temp first in case it already exists from a previous run
# this helps avoid an error when i create the temporary table again

cur.execute("DROP TABLE IF EXISTS metro_stats_temp;")


# then i create a temporary table called metro_stats_temp
# this table stores metro-level benchmark statistics that i will use later in the main query

# inside it, for each (city_full, date) group i calculate:
# - metro_median_price using PERCENTILE_CONT(0.5) 
# - metro_total_inventory using SUM(inventory) in that metro and period
# so one row in metro_stats_temp = one metro area in one time period


cur.execute("""
    CREATE TABLE metro_stats_temp AS
    SELECT
        city_full,
        date,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY price) AS metro_median_price,
        SUM(inventory) AS metro_total_inventory
    FROM combined_housets
    WHERE city_full IS NOT NULL
      AND date IS NOT NULL
      AND price IS NOT NULL
      AND inventory IS NOT NULL
    GROUP BY city_full, date;
""")

conn.commit()

# check

cur.execute("""
    SELECT *
    FROM metro_stats_temp
""")

data = cur.fetchall()
columns = [desc[0] for desc in cur.description]

df_show = pd.DataFrame(data=data, columns=columns)
df_show

Combine ZIP-level observations with metro-level benchmark values + calc ```price_premium``` and ```inventory_share ```

In [ ]:
cur.execute("DROP TABLE IF EXISTS zip_metrics_temp;")

# from combined_housets i take the ZIP-level variables:
# - date
# - zipcode
# - city_full
# - price
# - inventory

# from metro_stats_temp i add the metro-level benchmark variables:
# - metro_median_price
# - metro_total_inventory

# i join combined_housets with metro_stats_temp on city_full and date - each ZIP row gets the benchmark values of its own metro area in the same time period

# i calculate price_premium as ZIP price / metro median price
# and inventory_share as ZIP inventory / total metro inventory

# then i calculate prev_price using LAG(ch.price): LAG gives the price of the same ZIP code in the previous time period
# i use PARTITION BY ch.zipcode so that the previous price is calculated separately within each ZIP code


cur.execute("""
    CREATE TABLE zip_metrics_temp AS
    SELECT
        ch.date,
        ch.zipcode,
        ch.city_full,
        ch.price,
        ch.inventory,
        ms.metro_median_price,
        ms.metro_total_inventory,
      
        ch.price / NULLIF(ms.metro_median_price, 0) AS price_premium,
        ch.inventory::REAL / NULLIF(ms.metro_total_inventory, 0) AS inventory_share,
            
        LAG(ch.price) OVER (
            PARTITION BY ch.zipcode
            ORDER BY ch.date
        ) AS prev_price

    FROM combined_housets ch
    JOIN metro_stats_temp ms
      ON ch.city_full = ms.city_full
     AND ch.date = ms.date
    WHERE ch.city_full IS NOT NULL
      AND ch.zipcode IS NOT NULL
      AND ch.date IS NOT NULL
      AND ch.price IS NOT NULL
      AND ch.inventory IS NOT NULL;
""")

conn.commit()

# check

cur.execute("""
    SELECT *
    FROM zip_metrics_temp
""")

data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_show = pd.DataFrame(data=data, columns=columns)
df_show

Calculate ``price_momentum``

In [ ]:
cur.execute("DROP TABLE IF EXISTS zip_momentum_temp;")

# temporary table  zip_momentum_temp
# this table is based on zip_metrics_temp, which already contains ZIP-level variables, metro-level benchmarks, ratio metrics, and prev_price

# i calculate price_momentum as (price - prev_price) / prev_price
# this measures how much the ZIP price changed relative to its previous-period value


cur.execute("""
    CREATE TABLE zip_momentum_temp AS
    SELECT
        *,
        (price - prev_price) / NULLIF(prev_price, 0) AS price_momentum
        
    FROM zip_metrics_temp;
""")

conn.commit()

# check

cur.execute("""
    SELECT *
    FROM zip_momentum_temp
""")

data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_show = pd.DataFrame(data=data, columns=columns)
df_show

Calculate ```price_momentum_rank``` to identify which ZIP codes are leading or lagging the metro trend and ```metro_zip_count``` to know the size of each metro-date group so that later i can define bottom-ranked ZIP codes as laggards

In [ ]:
cur.execute("DROP TABLE IF EXISTS ranked_zip_temp;")

# temporary table ranked_zip_temp based on zip_momentum_temp

# here i add two new variables using window functions

# first, i calculate price_momentum_rank with RANK()
# i use PARTITION BY city_full and date, so the ranking is done separately within each metro area in each time period
# inside each such group, ZIP codes are ordered by price_momentum in descending order
# this means that the ZIP with the highest price momentum gets rank 1

# second, i calculate metro_zip_count with COUNT(*) OVER (PARTITION BY city_full, date) - the total number of ZIP observations in each metro-date group
# i will need it to identify the bottom 3 ZIP codes in its metro-date group by price momentum


cur.execute("""
    CREATE TABLE ranked_zip_temp AS
    SELECT
        *,
        RANK() OVER (PARTITION BY city_full, date ORDER BY price_momentum DESC) AS price_momentum_rank,

        COUNT(*) OVER (PARTITION BY city_full, date) AS metro_zip_count
    FROM zip_momentum_temp;
""")

conn.commit()

# check

cur.execute("""
    SELECT *
    FROM ranked_zip_temp
""")

data = cur.fetchall()
columns = [desc[0] for desc in cur.description]

df_show = pd.DataFrame(data=data, columns=columns)
df_show

Final step: i classify each ZIP code according to its position in the metro-level momentum ranking

In [ ]:
# here i take all variables from ranked_zip_temp, which already contains price_momentum_rank and metro_zip_count
# then i create a new categorical variable called metro_trend_position

# if price_momentum_rank <= 3 - label the ZIP as 'leading_metro_trend'
# if price_momentum_rank >= metro_zip_count - 2 - 'lagging_metro_trend'
# all remaining ZIP codes are labeled as 'middle_of_distribution'

cur.execute("""
    ALTER TABLE ranked_zip_temp
    ADD COLUMN IF NOT EXISTS metro_trend_position TEXT;
""")

cur.execute("""
    UPDATE ranked_zip_temp
    SET metro_trend_position = CASE
        WHEN price_momentum_rank <= 3 THEN 'leading_metro_trend'
        WHEN price_momentum_rank >= metro_zip_count - 2 THEN 'lagging_metro_trend'
        ELSE 'middle_of_distribution'
    END;
""")

conn.commit()

cur.execute("""
    SELECT *
    FROM ranked_zip_temp
    LIMIT 10
""")

# see final solution

data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_show = pd.DataFrame(data=data, columns=columns)
df_show

In [ ]:
columns

конец

In [ ]:
cur.execute("""CREATE TABLE final_version AS
            SELECT 
                ch.*,
                rzt.price_premium,
                rzt.inventory_share,
                rzt.price_momentum_rank,
                rzt.metro_trend_position
            FROM combined_housets ch
            JOIN ranked_zip_temp rzt
                ON ch.date = rzt.date
                AND ch.zipcode = rzt.zipcode
            """)

conn.commit()


теперь точно конец

In [ ]:
cur.execute("""SELECT * FROM final_version""")
columns = [desc[0] for desc in cur.description]
columns

69 

столбцов


а вы о чем подумали